In [ ]:
import sys, os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if BASE_DIR not in sys.path:
    sys.path.append(BASE_DIR)

from data_processing import MelDataManager

data_folder = "../../data/genres_original"
classes = ['blues', 'classical', 'country', 'disco', 'hiphop','jazz', 'metal', 'pop', 'reggae', 'rock']

In [ ]:
# # --- GPU + Mixed Precision Setup (start cell) ---
# from tensorflow.keras import mixed_precision

# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# gpus = tf.config.list_physical_devices('GPU')
# # if gpus:
# #     print(f"GPU detected: {gpus[0].name}")
# # else:
# #     print("GPU NOT detected — TensorFlow will run on CPU")
# if gpus:
#     try:
#         for gpu in gpus:
#             tf.config.experimental.set_memory_growth(gpu, True)
#     except RuntimeError as e:
#         print(e)

# mixed_precision.set_global_policy("mixed_float16")
# print("Mixed precision enabled (mixed_float16)")

# print(f"TensorFlow version: {tf.__version__}")
# print(f"Compute dtype: {mixed_precision.global_policy().compute_dtype}")
# print(f"Variable dtype: {mixed_precision.global_policy().variable_dtype}")

In [ ]:
mgr = MelDataManager(
    data_folder=data_folder,
    classes=classes,
    cache_path="../../data/processed/melspec_cache.npz",
    target_shape=(150, 150),
    chunk_duration=4,
    overlap_duration=2,
)

data, labels = mgr.load_or_build_cache()
print(f"Data shape: {data.shape}")
print(f"Labels shape (before one-hot): {labels.shape}")

In [ ]:
labels = mgr.one_hot()
print(f"Labels shape (one-hot): {labels.shape}")

In [ ]:
X_train, X_test, Y_train, Y_test = mgr.split(test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"Y_test shape: {Y_test.shape}")

In [ ]:
# Konwersja na RGB (MobileNetV2 wymaga 3 kanałów)
X_train_rgb = np.repeat(X_train, 3, axis=-1)
X_test_rgb = np.repeat(X_test, 3, axis=-1)
print(f"Przed RGB: {X_train.shape}")
print(f"Po RGB: {X_train_rgb.shape}")

In [ ]:
base_model = MobileNetV2(
    input_shape=(150, 150, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.2)(x)
# output = layers.Dense(10, activation='softmax')(x)
output = layers.Dense(10, activation='softmax', dtype='float32')(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
history = model.fit(
    X_train_rgb, Y_train,
    validation_split=0.2,
    epochs=30,
    batch_size=2,
    # batch_size=32,
    verbose=1
)

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(acc, label='Trening', color='blue')
ax1.plot(val_acc, label='Walidacja', color='red')
ax1.set_title('Transfer Learning Dokładność (Accuracy)')
ax1.set_xlabel('Epoka')
ax1.set_ylabel('Wartość')
ax1.legend()
ax1.grid(True)

ax2.plot(loss, label='Trening', color='blue')
ax2.plot(val_loss, label='Walidacja', color='red')
ax2.set_title('Transfer Learning Strata (Loss)')
ax2.set_xlabel('Epoka')
ax2.set_ylabel('Wartość')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Ewaluacja na zbiorze testowym
test_loss, test_acc = model.evaluate(X_test_rgb, Y_test, verbose=1)
print(f"\nDokładność na zbiorze testowym: {test_acc:.2%}")
print(f"Strata na zbiorze testowym: {test_loss:.4f}")

In [ ]:
Y_pred_proba = model.predict(X_test_rgb)
Y_pred_classes = np.argmax(Y_pred_proba, axis=1)
Y_true_classes = np.argmax(Y_test, axis=1)

cm = confusion_matrix(Y_true_classes, Y_pred_classes)
cm_display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)

fig, ax = plt.subplots(figsize=(10, 10))
cm_display.plot(cmap=plt.cm.Blues, ax=ax)
plt.title('Transfer Learning (MobileNetV2) - Macierz Pomyłek', fontsize=14)
plt.xticks(rotation=45)
plt.show()

print("\nRaport klasyfikacji:")
print(classification_report(Y_true_classes, Y_pred_classes, target_names=classes))

In [ ]:
model.save('../models/Transfer_learning.keras')
print("Model zapisany jako: ../models/Transfer_learning.keras")

In [ ]:
# # Fine-tuning (opcjonalnie - odmrożenie części warstw bazowych)
# print("Rozpoczynam fine-tuning...")

# # Odmroź ostatnie 30 warstw
# base_model.trainable = True
# for layer in base_model.layers[:-30]:
#     layer.trainable = False

# # Rekompiluj z mniejszą learning rate
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
#     loss='categorical_crossentropy',
#     metrics=['accuracy']
# )

# # Dotrenuj
# history_fine = model.fit(
#     X_train_rgb, Y_train,
#     validation_split=0.2,
#     epochs=10,
#     batch_size=32,
#     verbose=1
# )

In [ ]:
# # Ewaluacja po fine-tuningu
# test_loss_fine, test_acc_fine = model.evaluate(X_test_rgb, Y_test, verbose=1)
# print(f"\nDokładność po fine-tuningu: {test_acc_fine:.2%}")
# print(f"Poprawa: {(test_acc_fine - test_acc)*100:.2f} punktów procentowych")

In [ ]:
# # Zapisz model
# model.save('../models/transfer_learning_model.keras')
# print("Model zapisany jako: ../models/transfer_learning_model.keras")